<a href="https://colab.research.google.com/github/DeepFluxion/Mack_2026_Data_Science_Expirience/blob/main/notebooks/testes_hipoteses_bank_marketing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Testes de Hipóteses — Bank Marketing Dataset

**MBA em Data Science** | Disciplina: Estatística para Machine Learning  
**Dataset:** UCI Bank Marketing (`bank-additional-full.csv`) — 41.188 registros  
**Ambiente:** Google Colab (recomendado) ou Jupyter Notebook local  
**Objetivo:** Avaliar se as variáveis numéricas têm distribuições diferentes entre clientes que assinaram (`y=yes`) e que não assinaram (`y=no`) o depósito a prazo.

---

## 🎯 O que você vai aprender neste notebook

1. O que é um **teste de hipóteses** e por que usamos em Data Science
2. Como **escolher o teste correto** para cada situação
3. Como **interpretar p-values e effect sizes**
4. Por que o **tamanho do efeito** é mais importante que o p-value em grandes amostras
5. Como usar testes estatísticos para **priorizar features** de um modelo de ML

---

> **Contexto:** Um banco português realiza campanhas de telemarketing para vender depósitos a prazo.  
> **Pergunta central:** As variáveis numéricas se comportam de forma **diferente** para quem assinou vs quem não assinou?  
> Se sim → a variável tem potencial para ajudar um modelo de Machine Learning a fazer previsões.

---
# 📚 Seção 1 — Fundamentos dos Testes de Hipóteses

## 1.1 O que é um Teste de Hipóteses?

### Uma analogia simples: o tribunal

Imagine que você é um **juiz**. Ao início do julgamento, você parte do princípio que o réu é **inocente** — essa é a sua hipótese inicial.  
Ao longo do processo, as evidências são apresentadas. Se forem suficientemente fortes, você muda de opinião e condena o réu.

Em estatística, o raciocínio é **idêntico**:

| No Tribunal | Em Estatística |
|---|---|
| Réu é inocente (até prova em contrário) | **H₀ (Hipótese Nula):** as distribuições dos dois grupos são iguais |
| Réu é culpado | **H₁ (Hipótese Alternativa):** as distribuições são diferentes |
| Evidências apresentadas | Os dados do nosso dataset |
| Força das evidências | **p-value** |
| Veredito final | Rejeitar ou não H₀ |

---

## 1.2 H₀ e H₁ no Nosso Problema

Para cada variável numérica (ex: `age`), testamos:

- **H₀ (Hipótese Nula):** A idade dos clientes com `y=yes` e `y=no` tem a **mesma distribuição**  
  → Conclusão: a variável `age` **não ajuda** a prever quem vai assinar

- **H₁ (Hipótese Alternativa):** As distribuições são **diferentes**  
  → Conclusão: `age` **tem potencial** para ser usada no modelo preditivo

---

## 1.3 O que é o p-value?

O p-value responde à pergunta:

> *"Se H₀ fosse verdadeira (distribuições iguais), qual seria a probabilidade de eu observar uma diferença tão grande quanto a encontrada nos meus dados?"*

- **p-value < 0,05 (α):** A diferença observada seria muito improvável por acaso → **Rejeitamos H₀** ✅
- **p-value ≥ 0,05 (α):** A diferença pode ter ocorrido por acaso → **Não rejeitamos H₀** ❌

O valor **α = 0,05** é chamado de **nível de significância** — é o nosso limiar de decisão.

---

## 1.4 Tipos de Erros

| | **H₀ é verdadeira** (distribuições iguais) | **H₀ é falsa** (distribuições diferentes) |
|---|---|---|
| **Rejeitar H₀** | ❌ **Erro Tipo I** — falso positivo<br>"Condenar um inocente" | ✅ Decisão correta |
| **Não rejeitar H₀** | ✅ Decisão correta | ❌ **Erro Tipo II** — falso negativo<br>"Absolver um culpado" |

O nível de significância **α = 0,05** controla a taxa de Erro Tipo I — aceitamos 5% de chance de cometer esse erro.

## 1.5 Quais Testes Existem? — Guia Completo

A escolha do teste depende de uma pergunta fundamental: **os dados seguem uma distribuição normal?**

---

### 🔍 Passo 1: Testes de Normalidade

Antes de tudo, precisamos saber se os dados têm o formato de uma "curva de sino" (distribuição normal).

| Teste | Para que serve | Quando usar | Limitação |
|---|---|---|---|
| **Shapiro-Wilk** | Testa se os dados são normais | Amostras pequenas (**n < 50**) | Não funciona bem com n > 5.000 — p-value sempre < 0,05 |
| **Kolmogorov-Smirnov (KS)** | Compara a distribuição empírica com a teórica | Amostras grandes (**n > 5.000**) | Menos sensível que Shapiro-Wilk |
| **Anderson-Darling** | Versão aprimorada do KS, mais sensível nas caudas | Qualquer tamanho | Mais complexo de interpretar |
| **QQ-Plot** | Visualização rápida e intuitiva | **Sempre!** | Subjetivo — requer interpretação visual |

> 💡 **Como ler um QQ-Plot:** Se os pontos ficam próximos da linha diagonal → distribuição normal.  
> Desvios nas extremidades indicam caudas pesadas (assimetria ou outliers).

**Em nosso dataset (n = 41.188):** usaremos o **KS-test** para o teste formal e o **QQ-Plot** para visualização.

---

### ⚖️ Passo 2: Testes de Comparação (2 grupos)

Após verificar normalidade, escolhemos o teste de comparação:

| Teste | Tipo | Pré-requisito | O que compara | Quando usar |
|---|---|---|---|---|
| **Student's t-test** | Paramétrico | Normal + **variâncias iguais** | Médias | Raramente — pressuposto muito restritivo |
| **Welch's t-test** | Paramétrico | Normal + variâncias **podem ser diferentes** | Médias | **Preferido** quando dados são normais |
| **Mann-Whitney U** | Não-paramétrico | **Nenhum** — funciona com qualquer distribuição | Posições (rankings) | Quando dados **não são normais** |

> 💡 **Paramétrico vs Não-paramétrico:**
> - **Paramétrico** = assume que os dados têm um formato específico (normal). Mais poderoso quando o pressuposto é atendido.
> - **Não-paramétrico** = não assume nada sobre o formato dos dados. Mais robusto a outliers e assimetria.
> - O **Mann-Whitney U** não compara médias — ele compara se os valores de um grupo tendem a ser sistematicamente maiores que do outro.

---

### 📏 Passo 3: Tamanho do Efeito (*Effect Size*)

O p-value diz **se** há diferença. O effect size diz **o quanto** há de diferença. **Este é o mais importante!**

| Métrica | Usado com | Como interpretar | Limiares de Cohen |
|---|---|---|---|
| **Cohen's d** | Welch's t-test | Diferença entre médias em desvios padrão | d < 0,2 = negligível · 0,2–0,5 = pequeno · 0,5–0,8 = médio · > 0,8 = grande |
| **Rank-biserial r** | Mann-Whitney U | Probabilidade de um grupo ter valores maiores | r < 0,1 = negligível · 0,1–0,3 = pequeno · 0,3–0,5 = médio · > 0,5 = grande |

---

### ⚠️ A Armadilha das Amostras Grandes

Com **n = 41.188 registros**, o teste estatístico detectará diferenças **minúsculas** como "significativas" (p < 0,05).  
Uma diferença de 0,01 unidade pode ter p-value = 0,0001 — mas isso não significa que a variável é útil!

```
Exemplo real: variável X → p-value = 0,000001 → 'Rejeita H₀' ✅
                         → Cohen's d = 0,03   → Efeito NEGLIGÍVEL ⛔

Conclusão: a variável X tem diferença estatisticamente significativa,
           mas o efeito é tão pequeno que não ajudará o modelo a prever nada.
```

---

### 🗺️ Fluxo de Decisão

```
     Variável numérica
            │
            ▼
   Testar normalidade (KS-test)
            │
     ┌──────┴──────┐
  Normal         Não-normal
     │                │
     ▼                ▼
 Welch's t-test  Mann-Whitney U
     │                │
     ▼                ▼
  Cohen's d    Rank-biserial r
     │                │
     └──────┬──────────┘
            ▼
     Interpretar magnitude
     (negligível/pequeno/médio/grande)
            │
            ▼
     Decisão: relevante para ML?
```

---
# 🐍 Seção 2 — Estrutura de um Teste de Hipóteses em Python

Agora vamos **traduzir o fluxo de decisão em código Python**.  
A abordagem é modular: cada etapa do pipeline será uma função separada,  
o que facilita a reutilização e a leitura do código.

**Pipeline de funções:**
```
verificar_normalidade()      → KS-test para cada grupo
calcular_cohen_d()           → effect size paramétrico
calcular_rank_biserial()     → effect size não-paramétrico
interpretar_effect_size()    → classifica a magnitude
executar_teste_hipotese()    → orquestra todo o pipeline
```

In [ ]:
# ─── Bibliotecas ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind, kstest
import warnings
warnings.filterwarnings('ignore')

# ─── Configurações visuais ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False
})

# ─── Constantes ─────────────────────────────────────────────────────────────────
COR_NO    = '#378ADD'   # Azul    → y = no
COR_YES   = '#E24B4A'   # Vermelho → y = yes
ALPHA     = 0.05        # Nível de significância

print('Versões:')
print(f'  pandas  {pd.__version__}')
print(f'  numpy   {np.__version__}')
print('Configurações carregadas!')

In [ ]:
import io
import os

# ─── Upload do arquivo (Google Colab) ──────────────────────────────────────────
try:
    from google.colab import files

    print('Ambiente Google Colab detectado!')
    print()
    print('Por favor, faca o upload do arquivo: bankadditionalfull.csv')
    print('(clique em "Escolher arquivos" abaixo)')
    print()

    uploaded = files.upload()   # Abre o seletor de arquivo

    if not uploaded:
        raise ValueError('Nenhum arquivo enviado. Execute a celula novamente e selecione o arquivo.')

    filename = list(uploaded.keys())[0]
    print(f'Arquivo recebido: {filename} ({len(uploaded[filename]):,} bytes)')
    df = pd.read_csv(io.BytesIO(uploaded[filename]), sep=';')

except ImportError:
    # ─── Fallback: execucao local (fora do Colab) ──────────────────────────────
    print('Ambiente local detectado — carregando arquivo diretamente...')
    df = pd.read_csv('bankadditionalfull.csv', sep=';')

# ─── Verificacoes ──────────────────────────────────────────────────────────────
print()
print(f'Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
print()

# Distribuicao do target
print('Distribuicao do target (y):')
counts = df['y'].value_counts()
pcts   = df['y'].value_counts(normalize=True) * 100
for lbl in ['no', 'yes']:
    barra = chr(9608) * int(pcts[lbl] / 2)
    print(f'  {lbl:3s}  {counts[lbl]:6,}  ({pcts[lbl]:5.1f}%)  {barra}')

print()
print('Desbalanceamento: proporcao ~1:8 (yes:no)')
print()

# Lista de variaveis numericas
VARS_NUM = ['age', 'duration', 'campaign', 'pdays', 'previous',
            'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
            'euribor3m', 'nr.employed']

print(f'{len(VARS_NUM)} variaveis numericas:')
for v in VARS_NUM:
    print(f'  - {v}')

df[VARS_NUM + ['y']].head()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FUNCOES AUXILIARES — Pipeline de Teste de Hipoteses
# ══════════════════════════════════════════════════════════════════════════════

def estatisticas_por_grupo(df, variavel):
    """Retorna estatisticas descritivas separadas por grupo (y=no, y=yes e total)."""
    resultados = {}
    for grupo in ['no', 'yes', 'total']:
        dados = df[variavel].dropna() if grupo == 'total' else df[df['y'] == grupo][variavel].dropna()
        chave = 'total' if grupo == 'total' else f'y={grupo}'
        resultados[chave] = {
            'n':            len(dados),
            'media':        round(dados.mean(), 3),
            'mediana':      round(dados.median(), 3),
            'desv_padrao':  round(dados.std(), 3),
            'minimo':       round(dados.min(), 3),
            'maximo':       round(dados.max(), 3),
            'assimetria':   round(dados.skew(), 3),
            'curtose':      round(dados.kurtosis(), 3)
        }
    return pd.DataFrame(resultados)


def verificar_normalidade(dados, alpha=ALPHA):
    """
    Teste de normalidade usando KS-test (Kolmogorov-Smirnov).

    Como funciona:
    - Ajusta uma distribuicao normal aos dados (mesma media e desvio padrao)
    - Mede a maior diferenca entre a distribuicao empirica e a teorica
    - p-value pequeno = a diferenca e grande = nao e normal

    Retorna: (estatistica, p-value, eh_normal)
    """
    media, std = dados.mean(), dados.std()
    stat, pval = kstest(dados, 'norm', args=(media, std))
    return stat, pval, (pval > alpha)


def calcular_cohen_d(g1, g2):
    """
    Cohen's d — effect size para testes parametricos (t-test).

    Interpretacao: mede quantos desvios padrao separam as duas medias.
    - d = 0,5 significa: as medias diferem em 0,5 desvios padrao

    Formula: d = (media1 - media2) / desvio_padrao_pooled
    """
    n1, n2 = len(g1), len(g2)
    s_pooled = np.sqrt(((n1 - 1) * g1.std()**2 + (n2 - 1) * g2.std()**2) / (n1 + n2 - 2))
    return abs((g1.mean() - g2.mean()) / s_pooled) if s_pooled > 0 else 0.0


def calcular_rank_biserial(u_stat, n1, n2):
    """
    Rank-biserial r — effect size para Mann-Whitney U.

    Interpretacao: probabilidade de que um valor aleatorio do grupo 1
    seja maior que um valor aleatorio do grupo 2.
    - r = 0: sem diferenca entre grupos
    - r = 1: todos os valores do grupo 1 sao maiores que todos do grupo 2

    Formula: r = |1 - (2 * U) / (n1 * n2)|
    """
    return abs(1 - (2 * u_stat) / (n1 * n2))


def interpretar_effect_size(valor, tipo='r'):
    """
    Classifica o tamanho do efeito segundo as convencoes de Cohen.
    tipo='d' para Cohen's d | tipo='r' para rank-biserial
    """
    if tipo == 'd':
        if valor < 0.2:   return 'Negligivel'
        elif valor < 0.5: return 'Pequeno'
        elif valor < 0.8: return 'Medio'
        else:             return 'Grande'
    else:  # rank-biserial r
        if valor < 0.1:   return 'Negligivel'
        elif valor < 0.3: return 'Pequeno'
        elif valor < 0.5: return 'Medio'
        else:             return 'Grande'


def executar_teste_hipotese(df, variavel, alpha=ALPHA, verbose=True):
    """
    Pipeline completo de teste de hipotese para uma variavel numerica.

    Passo 1 — Normalidade: KS-test em cada grupo (y=no e y=yes)
    Passo 2 — Teste de comparacao:
        Ambos normais  → Welch's t-test + Cohen's d
        Nao-normais    → Mann-Whitney U + Rank-biserial r
    Passo 3 — Interpretacao da magnitude do efeito

    Retorna dicionario com todos os resultados.
    """
    g_no  = df[df['y'] == 'no'][variavel].dropna()
    g_yes = df[df['y'] == 'yes'][variavel].dropna()

    # Passo 1: Normalidade
    _, pval_no,  norm_no  = verificar_normalidade(g_no,  alpha)
    _, pval_yes, norm_yes = verificar_normalidade(g_yes, alpha)
    ambos_normais = norm_no and norm_yes

    # Passo 2: Teste de comparacao
    if ambos_normais:
        stat, pval  = ttest_ind(g_no, g_yes, equal_var=False)  # Welch
        teste        = "Welch's t-test"
        es           = calcular_cohen_d(g_no, g_yes)
        es_tipo      = 'd'
        es_nome      = "Cohen's d"
    else:
        stat, pval  = mannwhitneyu(g_no, g_yes, alternative='two-sided')
        teste        = 'Mann-Whitney U'
        es           = calcular_rank_biserial(stat, len(g_no), len(g_yes))
        es_tipo      = 'r'
        es_nome      = 'Rank-biserial r'

    # Passo 3: Interpretacao
    magnitude  = interpretar_effect_size(es, es_tipo)
    rejeita_h0 = pval < alpha

    resultado = {
        'variavel':         variavel,
        'norm_no':          norm_no,
        'norm_yes':         norm_yes,
        'teste':            teste,
        'estatistica':      round(float(stat), 4),
        'p_value':          pval,
        'rejeita_h0':       rejeita_h0,
        'effect_size_nome': es_nome,
        'effect_size':      round(es, 4),
        'magnitude':        magnitude
    }

    if verbose:
        _print_resultado(resultado)

    return resultado


def _print_resultado(r):
    """Formata e imprime o resultado do teste de hipotese."""
    emoji = {'Negligivel': '[o]', 'Pequeno': '[~]', 'Medio': '[!]', 'Grande': '[***]'}
    print('\n' + '=' * 58)
    print(f'  RESULTADO: {r["variavel"].upper()}')
    print('=' * 58)
    print('  NORMALIDADE (KS-test, alfa=0.05):')
    print(f'    y=no  -> {"Normal" if r["norm_no"]  else "Nao-normal"}')
    print(f'    y=yes -> {"Normal" if r["norm_yes"] else "Nao-normal"}')
    print(f'\n  TESTE ESCOLHIDO: {r["teste"]}')
    print(f'    Estatistica : {r["estatistica"]:.4f}')
    print(f'    p-value     : {r["p_value"]:.2e}')
    print(f'\n  DECISAO (alfa = {ALPHA}):')
    if r['rejeita_h0']:
        print('    Rejeita H0 -> Distribuicoes DIFERENTES (p < alfa)')
    else:
        print('    Nao rejeita H0 -> Distribuicoes similares (p >= alfa)')
    print(f'\n  TAMANHO DO EFEITO ({r["effect_size_nome"]}):')
    mag = r['magnitude']
    print(f'    Valor: {r["effect_size"]:.4f}  ->  {emoji[mag]} {mag}')
    print('\n  RELEVANCIA PARA ML:')
    if mag in ('Grande', 'Medio'):
        print('    [***] Alto poder discriminante -> manter no modelo')
    elif mag == 'Pequeno':
        print('    [ ~ ] Poder moderado -> avaliar em combinacao com outras vars')
    else:
        print('    [ o ] Baixo poder discriminante -> baixa prioridade')
    print('=' * 58)


print('Funcoes auxiliares carregadas!')
print('  - estatisticas_por_grupo(df, variavel)')
print('  - verificar_normalidade(dados)')
print('  - calcular_cohen_d(g1, g2)')
print('  - calcular_rank_biserial(u_stat, n1, n2)')
print('  - interpretar_effect_size(valor, tipo)')
print('  - executar_teste_hipotese(df, variavel)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FUNCOES DE VISUALIZACAO
# ══════════════════════════════════════════════════════════════════════════════

def visualizar_distribuicao(df, variavel, titulo_extra='', bins=40):
    """
    Gera 3 graficos complementares para analisar a distribuicao de uma variavel:
      Painel 1 — Histograma + KDE: mostra a 'forma' da distribuicao em cada grupo
      Painel 2 — Boxplot: compara mediana, dispersao e outliers
      Painel 3 — QQ-Plot: verifica visualmente se a distribuicao e normal
    """
    g_no  = df[df['y'] == 'no'][variavel].dropna()
    g_yes = df[df['y'] == 'yes'][variavel].dropna()

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    titulo = f"Analise de '{variavel}'{titulo_extra}"
    fig.suptitle(titulo, fontsize=14, fontweight='bold', y=1.01)

    # ── Painel 1: Histograma + KDE ─────────────────────────────────────────────
    ax1 = axes[0]
    sns.histplot(g_no,  bins=bins, alpha=0.45, color=COR_NO,
                 stat='density', label=f'y=no  (n={len(g_no):,})',  ax=ax1)
    sns.histplot(g_yes, bins=bins, alpha=0.45, color=COR_YES,
                 stat='density', label=f'y=yes (n={len(g_yes):,})', ax=ax1)
    sns.kdeplot(g_no,  color=COR_NO,  linewidth=2.5, ax=ax1)
    sns.kdeplot(g_yes, color=COR_YES, linewidth=2.5, ax=ax1)
    ax1.set_title('Histograma + KDE\n(separacao das curvas = poder discriminante)',
                  fontsize=10, fontweight='bold')
    ax1.set_xlabel(variavel)
    ax1.set_ylabel('Densidade')
    ax1.legend(fontsize=9)

    # ── Painel 2: Boxplot ──────────────────────────────────────────────────────
    ax2 = axes[1]
    dados_box = pd.DataFrame({
        'Valor':  pd.concat([g_no, g_yes], ignore_index=True),
        'Grupo': ['y = no'] * len(g_no) + ['y = yes'] * len(g_yes)
    })
    sns.boxplot(data=dados_box, x='Grupo', y='Valor',
                palette={'y = no': COR_NO, 'y = yes': COR_YES},
                width=0.45, linewidth=1.5, fliersize=2, ax=ax2)
    # Adicionar marcador de media
    for i, (grp, dados_g) in enumerate([(g_no, g_no), (g_yes, g_yes)]):
        ax2.plot(i, dados_g.mean(), 'D', color='white', markersize=7,
                 zorder=5, markeredgecolor='black', markeredgewidth=1.2)
    ax2.set_title('Boxplot Comparativo\n(linha = mediana, diamante = media)',
                  fontsize=10, fontweight='bold')
    ax2.set_xlabel('')
    ax2.set_ylabel(variavel)

    # ── Painel 3: QQ-Plot ──────────────────────────────────────────────────────
    ax3 = axes[2]
    for dados, cor, lbl in [(g_no, COR_NO, 'y=no'), (g_yes, COR_YES, 'y=yes')]:
        res = stats.probplot(dados, dist='norm')
        osm, osr = res[0]
        slope, intercept, _ = res[1]
        ax3.scatter(osm, osr, color=cor, alpha=0.2, s=4, label=lbl)
        x_line = np.array([osm.min(), osm.max()])
        ax3.plot(x_line, slope * x_line + intercept, color=cor, linewidth=2)
    ax3.set_title('QQ-Plot (Normalidade)\n(pontos na linha = distribuicao normal)',
                  fontsize=10, fontweight='bold')
    ax3.set_xlabel('Quantis Teoricos (Normal)')
    ax3.set_ylabel('Quantis Observados')
    ax3.legend(fontsize=9)
    ax3.annotate('Se os pontos seguirem\na linha diagonal:\ndistribuicao normal',
                 xy=(0.03, 0.97), xycoords='axes fraction',
                 fontsize=8, va='top', color='gray',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    plt.tight_layout()
    plt.show()


def analisar_variavel(df, variavel, bins=40, titulo_extra=''):
    """Funcao completa: estatisticas descritivas + visualizacoes."""
    print(f'ESTATISTICAS DESCRITIVAS — {variavel.upper()}')
    print('-' * 55)
    print(estatisticas_por_grupo(df, variavel).to_string())
    print()
    visualizar_distribuicao(df, variavel, titulo_extra=titulo_extra, bins=bins)


print('Funcoes de visualizacao carregadas!')
print('  - visualizar_distribuicao(df, variavel)')
print('  - analisar_variavel(df, variavel)')

## 2.1 Demonstração do Pipeline — Passo a Passo com `age`

Antes de analisar todas as variáveis em loop, vamos demonstrar **cada passo do pipeline manualmente** com a variável `age`.  
Isso deixa o fluxo de decisão completamente transparente.

In [ ]:
# ── Demonstracao manual do pipeline com 'age' ─────────────────────────────────
print('DEMONSTRACAO DO PIPELINE — variavel: age')
print('=' * 55)

g_no  = df[df['y'] == 'no']['age'].dropna()
g_yes = df[df['y'] == 'yes']['age'].dropna()

print(f'\n  Grupo y=no : {len(g_no):,} observacoes')
print(f'  Grupo y=yes: {len(g_yes):,} observacoes')

# ── Passo 1: Normalidade ───────────────────────────────────────────────────────
print('\n--- PASSO 1: Testar Normalidade (KS-test) ---')
stat_no,  pval_no,  norm_no  = verificar_normalidade(g_no)
stat_yes, pval_yes, norm_yes = verificar_normalidade(g_yes)

print(f'  y=no  -> KS={stat_no:.4f}, p-value={pval_no:.4f} -> {"Normal" if norm_no else "Nao-normal"}')
print(f'  y=yes -> KS={stat_yes:.4f}, p-value={pval_yes:.4f} -> {"Normal" if norm_yes else "Nao-normal"}')

# ── Passo 2: Escolha do teste ──────────────────────────────────────────────────
print('\n--- PASSO 2: Escolher o Teste de Comparacao ---')
if norm_no and norm_yes:
    print('  Ambos normais -> Welch\'s t-test')
    stat_t, pval_t = ttest_ind(g_no, g_yes, equal_var=False)
    print(f'  t-estatistica = {stat_t:.4f} | p-value = {pval_t:.4e}')
    es_val  = calcular_cohen_d(g_no, g_yes)
    es_nome = "Cohen's d"
    es_tipo = 'd'
else:
    print('  Nao-normal -> Mann-Whitney U')
    stat_u, pval_u = mannwhitneyu(g_no, g_yes, alternative='two-sided')
    print(f'  U-estatistica = {stat_u:.0f} | p-value = {pval_u:.4e}')
    es_val  = calcular_rank_biserial(stat_u, len(g_no), len(g_yes))
    es_nome = 'Rank-biserial r'
    es_tipo = 'r'

# ── Passo 3: Effect size ───────────────────────────────────────────────────────
print(f'\n--- PASSO 3: Calcular Effect Size ---')
mag = interpretar_effect_size(es_val, es_tipo)
print(f'  {es_nome} = {es_val:.4f} -> Magnitude: {mag}')

# ── Verificacao com a funcao completa ──────────────────────────────────────────
print('\n--- VERIFICACAO: usando executar_teste_hipotese() ---')
_ = executar_teste_hipotese(df, 'age', verbose=True)

---
# 📈 Seções 3 e 4 — Análise Individual das Variáveis Numéricas

Para cada variável, apresentamos:
- **Contexto:** o que mede e por que importa
- **Estatísticas descritivas** comparando y=no, y=yes e total
- **3 gráficos:** Histograma+KDE, Boxplot, QQ-Plot

> 🎨 **Legenda de cores:** 🔵 Azul = `y=no` (não assinou) · 🔴 Vermelho = `y=yes` (assinou)

---
## 👤 Grupo 1 — Dados do Cliente

### Variável: `age` — Idade do cliente

**O que mede:** Idade em anos do cliente contactado.

**Hipótese prévia:** Clientes em fases específicas de vida tendem a ter perfil financeiro diferente:  
- **Jovens (< 30):** estudantes, maior flexibilidade, abertos a produtos financeiros  
- **Meia idade (30–60):** compromissos financeiros maiores (casa, filhos), menos propensos  
- **Idosos (> 60):** aposentados, renda estável, maior propensão a depósitos conservadores  

**Esperado:** Efeito não-linear — jovens e idosos com taxas de conversão mais altas que adultos de meia-idade.

In [ ]:
analisar_variavel(df, 'age', bins=35)

---
### ⚠️ Variável: `duration` — Duração da última ligação (em segundos)

**O que mede:** Tempo em segundos da última chamada telefônica ao cliente.

---

> ## 🚨 ALERTA CRÍTICO: DATA LEAKAGE — LEIA ANTES DE PROSSEGUIR!
>
> **Data Leakage** acontece quando o modelo usa informações que, na realidade, **não estariam disponíveis no momento da previsão**.
>
> **Por que `duration` é um caso clássico de leakage?**
>
> | Situação | Valor de duration | Resultado (y) |
> |---|---|---|
> | Cliente nunca atendeu | 0 segundos | **Sempre `no`** |
> | Ligação breve, cliente recusou | Poucos segundos | Quase sempre `no` |
> | Ligação longa, cliente negociando | Muitos segundos | Tendência a `yes` |
>
> **O problema:** Você só sabe a duração da chamada **depois que ela terminou** —  
> ou seja, **depois que o resultado `y` já foi determinado!**
>
> Incluir `duration` no modelo é como usar o gabarito da prova para respondê-la.  
> O modelo vai parecer ter desempenho perfeito, mas **não funcionará na prática**.
>
> ✅ **Por que analisamos aqui?** Para fins **didáticos**:  
> `duration` ilustra como uma variável com effect size GIGANTE pode ser completamente inútil para o negócio.  
> **Esta variável deve ser excluída de qualquer modelo preditivo realista.**


In [ ]:
analisar_variavel(df, 'duration', bins=50,
                  titulo_extra=' [DATA LEAKAGE - nao usar em modelos!]')

---
## 📞 Grupo 2 — Dados da Campanha Atual e Histórico

### Variável: `campaign` — Número de contatos nesta campanha

**O que mede:** Quantas vezes o cliente foi contactado durante **esta** campanha (inclui o último contato).

**Hipótese prévia:** Há um efeito de **retornos decrescentes** — ligar muitas vezes para o mesmo cliente tende a irritá-lo, e clientes que disseram `yes` provavelmente decidiram mais rapidamente (menos ligações necessárias).

**Esperado:** Clientes com `y=yes` foram contactados menos vezes em média.

In [ ]:
analisar_variavel(df, 'campaign', bins=20)

### Variável: `pdays` — Dias desde o último contato em campanha anterior

**O que mede:** Número de dias que passaram desde que o cliente foi contactado em uma **campanha anterior**.

**⚠️ Codificação especial:** O valor **999 significa que o cliente NUNCA foi contactado antes**.  
Isso **não é um número real** — é uma codificação de ausência de informação (como um `NaN` mascarado).

Isso cria uma distribuição muito estranha:  
- Enorme concentração em 999 (~96% dos registros não têm contato anterior)
- Uma pequena cauda com valores reais (0 a ~30 dias)

Analisaremos as duas perspectivas:
1. **Distribuição completa** (com 999) — mostra a realidade dos dados
2. **Distribuição filtrada** (sem 999) — analisa apenas quem teve contato anterior

In [ ]:
# ── Distribuicao COMPLETA (com 999) ───────────────────────────────────────────
print('=== Distribuicao COMPLETA (inclui 999 = sem contato anterior) ===')
analisar_variavel(df, 'pdays', bins=30)

# ── Distribuicao FILTRADA (excluindo 999) ──────────────────────────────────────
print('\n=== Distribuicao FILTRADA (somente clientes contactados antes) ===')
df_pdays = df[df['pdays'] != 999].copy()
n_filtrado = len(df_pdays)
pct_yes = (df_pdays['y'] == 'yes').mean() * 100
print(f'  Registros apos filtro: {n_filtrado:,} ({n_filtrado/len(df)*100:.1f}% do total)')
print(f'  Taxa y=yes no subgrupo: {pct_yes:.1f}% (vs 11.3% na base total)')
print('  -> Clientes com contato anterior convertem muito mais!')
print()
analisar_variavel(df_pdays, 'pdays', bins=25,
                  titulo_extra=' (excluindo 999 = sem contato anterior)')

### Variável: `previous` — Contatos em campanhas anteriores

**O que mede:** Número de vezes que o cliente foi contactado **antes desta campanha** (em qualquer campanha anterior).

**Hipótese prévia:** Clientes com histórico de engajamento já conhecem o banco, têm mais confiança e são mais propensos a assinar.

**Relacionada com `pdays`:** Se `pdays=999` (nunca contactado), então `previous=0` necessariamente.

In [ ]:
analisar_variavel(df, 'previous', bins=15)

---
## 🌍 Grupo 3 — Indicadores Socioeconômicos

Estas 5 variáveis capturam o **contexto macroeconômico de Portugal** no momento do contato:

| Variável | O que mede | Frequência de atualização |
|---|---|---|
| `emp.var.rate` | Taxa de variação do emprego — economia aquecendo ou esfriando? | Trimestral |
| `cons.price.idx` | Índice de preços ao consumidor — inflação | Mensal |
| `cons.conf.idx` | Índice de confiança do consumidor — otimismo da população | Mensal |
| `euribor3m` | Taxa Euribor 3 meses — custo do dinheiro na Europa | Diária |
| `nr.employed` | Número de empregados (em milhares) — tamanho do mercado de trabalho | Trimestral |

**Hipótese prévia:** Estas variáveis são altamente correlacionadas entre si (refletem o mesmo ciclo econômico) e provavelmente refletem que o banco intensificou as campanhas em períodos de baixas taxas de juros (quando o depósito a prazo é mais atraente para captar liquidez).

**Esperado:** Efeito grande — o contexto econômico deve discriminar bem os grupos.

### Variável: `emp.var.rate` — Taxa de variação do emprego

Indicador trimestral. Valores positivos = emprego crescendo (economia aquecida). Valores negativos = desemprego aumentando (recessão).  
**Hipótese:** Em recessão (valores negativos), o banco incentiva mais depósitos para captar liquidez — maior conversão.

In [ ]:
analisar_variavel(df, 'emp.var.rate', bins=20)

### Variável: `cons.price.idx` — Índice de Preços ao Consumidor

Mede a inflação mensal. Valores altos = inflação alta.  
**Hipótese:** Períodos de inflação elevada reduzem o poder aquisitivo e podem afetar a disposição a investir.

In [ ]:
analisar_variavel(df, 'cons.price.idx', bins=20)

### Variável: `cons.conf.idx` — Índice de Confiança do Consumidor

Mede o otimismo dos consumidores com a economia. Valores negativos são comuns (escala relativa).  
**Hipótese:** Maior confiança → maior disposição a comprometer dinheiro em investimentos de longo prazo.

In [ ]:
analisar_variavel(df, 'cons.conf.idx', bins=20)

### Variável: `euribor3m` — Taxa Euribor 3 meses

Taxa de referência do mercado monetário europeu. Define o custo do dinheiro.  
**Hipótese:** Quando a Euribor é baixa, os depósitos a prazo ficam mais atrativos (concorrência com outras aplicações diminui) → maior conversão. Esta deve ser uma das variáveis mais discriminantes.

In [ ]:
analisar_variavel(df, 'euribor3m', bins=25)

### Variável: `nr.employed` — Número de empregados (em milhares)

Indicador trimestral do mercado de trabalho português.  
**Hipótese:** Altamente correlacionada com `emp.var.rate` e `euribor3m` (todas refletem o ciclo econômico). Quando o emprego está baixo, o banco busca mais captação via depósitos.

In [ ]:
analisar_variavel(df, 'nr.employed', bins=20)

---
# 🧪 Seção 5 — Testes de Hipóteses: Execução e Resultados

Agora executamos o pipeline completo para **todas as variáveis numéricas** de uma vez.  
O resultado de cada teste segue exatamente o fluxo de decisão apresentado na Seção 1.

> **Lembrete:** Com n=41.188, o p-value será < 0,05 para praticamente todos os testes.  
> O critério real de importância é o **effect size** (tamanho do efeito).

```
Magnitude do Efeito:
  [o]   Negligivel  →  nenhuma diferenca pratica
  [~]   Pequeno     →  diferenca pequena, avaliar com cautela
  [!]   Medio       →  diferenca relevante, variavel util
  [***] Grande      →  alta diferenca, variavel muito util
```

In [ ]:
# ── Executar testes para todas as variaveis numericas ─────────────────────────
# Nota: duration e incluida com fins educacionais (NAO usar em producao!)

resultados = []
for variavel in VARS_NUM:
    r = executar_teste_hipotese(df, variavel, verbose=True)
    resultados.append(r)

In [ ]:
# ── Tabela resumo dos resultados ──────────────────────────────────────────────
tabela = pd.DataFrame(resultados)

# Formatacoes para exibicao
tabela['p_value_fmt'] = tabela['p_value'].apply(
    lambda x: f'{x:.2e}' if x < 0.001 else f'{x:.4f}'
)
tabela['decisao'] = tabela['rejeita_h0'].map(
    {True: 'Rejeita H0', False: 'Nao rejeita'}
)
tabela['obs'] = tabela['variavel'].map(
    lambda v: '[LEAKAGE]' if v == 'duration' else ''
)

# Ordenar por effect size
tabela_disp = (
    tabela[['variavel', 'teste', 'p_value_fmt', 'decisao',
             'effect_size_nome', 'effect_size', 'magnitude', 'obs']]
    .sort_values('effect_size', ascending=False)
    .reset_index(drop=True)
)
tabela_disp.columns = ['Variavel', 'Teste Usado', 'p-value', 'Decisao',
                        'Metrica ES', 'Valor ES', 'Magnitude', 'Obs']

print('TABELA RESUMO — Resultados dos Testes de Hipoteses')
print('(Ordenada por Effect Size decrescente)')
print()
try:
    display(tabela_disp)
except NameError:
    print(tabela_disp.to_string(index=False))

In [ ]:
# ── Ranking visual dos effect sizes ───────────────────────────────────────────
tabela_plot = tabela.sort_values('effect_size', ascending=True).copy()

cor_mag = {
    'Negligivel': '#D3D1C7',
    'Pequeno':    '#FAC775',
    'Medio':      '#EF9F27',
    'Grande':     '#E24B4A'
}
cores = [cor_mag[m] for m in tabela_plot['magnitude']]

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('white')

barras = ax.barh(tabela_plot['variavel'], tabela_plot['effect_size'],
                  color=cores, edgecolor='white', linewidth=0.5, height=0.65)

# Valores nas barras
for bar, val, mag in zip(barras, tabela_plot['effect_size'], tabela_plot['magnitude']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f} ({mag})', va='center', ha='left', fontsize=10.5)

# Linhas de referencia (limiares de Cohen para rank-biserial)
for x, lbl in [(0.1, 'Pequeno'), (0.3, 'Medio'), (0.5, 'Grande')]:
    ax.axvline(x=x, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.text(x + 0.003, len(tabela_plot) - 0.4, lbl, fontsize=8, color='gray', style='italic')

# Destacar duration (leakage)
ytick_labels = ax.get_yticklabels()
for label in ytick_labels:
    if label.get_text() == 'duration':
        label.set_color('#E24B4A')
        label.set_fontweight('bold')

# Legenda
legenda = [Patch(facecolor=c, label=m) for m, c in cor_mag.items()]
ax.legend(handles=legenda, title='Magnitude', loc='lower right', fontsize=9)

ax.set_title(
    'Ranking de Poder Discriminante — Effect Size por Variavel\n'
    '[duration em vermelho = DATA LEAKAGE, excluir do modelo]',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Effect Size (Rank-biserial r ou Cohen\'s d)')
ax.set_xlim(0, tabela_plot['effect_size'].max() * 1.38)
ax.set_facecolor('#f8f9fa')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
# 📝 Seção 6 — Conclusões e Próximos Passos

## 6.1 Síntese dos Resultados

### Variáveis por nível de poder discriminante

| Magnitude | Variáveis | Implicação para ML |
|---|---|---|
| 🔴 **Grande** | `euribor3m`, `nr.employed`, `emp.var.rate`, `cons.price.idx` | Manter — alto potencial preditivo |
| 🟠 **Médio** | `cons.conf.idx`, `pdays`, `previous`, `duration`* | Manter (exceto duration) |
| 🟡 **Pequeno** | `age`, `campaign` | Avaliar — úteis em combinação |
| ⚪ **Negligível** | — | Baixa prioridade |

*`duration` tem efeito grande mas deve ser **excluída** por data leakage

---

## 6.2 A Lição Mais Importante

> ### Significância Estatística ≠ Relevância Prática
>
> Com **41.188 registros**, todos os testes rejeitaram H₀ (p < 0,05).  
> Se usássemos apenas o p-value como critério, concluiríamos que todas as variáveis são igualmente importantes — o que é falso!
>
> **O effect size é o critério correto para priorizar features em grandes datasets.**

---

## 6.3 Cuidados Especiais por Variável

| Variável | Problema identificado | Ação recomendada no preprocessing |
|---|---|---|
| `duration` | **DATA LEAKAGE** | **Excluir completamente do modelo** |
| `pdays` | 999 = ausência de dado (não é número real) | Criar variável binária `contactado_antes` (0/1) + tratar 999 como NaN |
| `campaign` | Assimetria forte, outliers | Transformação log(x+1) |
| `previous` | Assimetria forte (maioria = 0) | Binarizar: 0 vs >0 |
| `age` | Efeito não-linear (jovens E idosos convertem mais) | Engenharia de features: criar faixas etárias |

---

## 6.4 Próximos Passos do Projeto

1. **Excluir `duration`** de todos os modelos preditivos
2. **Tratar `pdays=999`** — criar feature binária `contactado_anteriormente`
3. **Tratar valores `unknown`** nas variáveis categóricas (imputação ou remoção)
4. **Transformações** — `campaign`, `previous` → log(x+1) para reduzir assimetria
5. **Encoding** das variáveis categóricas (one-hot, ordinal conforme o tipo)
6. **Scaling** — StandardScaler para variáveis com escalas muito diferentes
7. **Balanceamento** — proporção 1:8 (yes:no) → SMOTE ou class_weight
8. **Modelagem** — Regressão Logística (baseline) → Random Forest → XGBoost

---

## 6.5 Checklist de Aprendizado

- [ ] Compreendo a diferença entre H₀ e H₁
- [ ] Sei interpretar um p-value e o nível de significância α
- [ ] Entendo quando usar Shapiro-Wilk vs KS-test para normalidade
- [ ] Sei quando escolher Welch's t-test vs Mann-Whitney U
- [ ] Compreendo por que effect size é mais importante que p-value em amostras grandes
- [ ] Identifiquei e sei explicar o problema de data leakage em `duration`
- [ ] Entendo como tratar `pdays=999` como ausência de dado
- [ ] Consigo priorizar features para ML com base em evidências estatísticas

---

*📚 Referências:*  
*Cohen, J. (1988). Statistical Power Analysis for the Behavioral Sciences. 2nd ed. Lawrence Erlbaum.*  
*Moro et al. (2014). A Data-Driven Approach to Predict the Success of Bank Telemarketing. Decision Support Systems.*  
*Mann, H.B. & Whitney, D.R. (1947). On a Test of Whether One of Two Random Variables is Stochastically Larger than the Other. Annals of Mathematical Statistics.*